In [1]:
%%writefile utils.py
import pandas as pd
import re

# 缩略词映射字典
CONTRACTION_MAPPING = {
    "ain't": "is not", "aren't": "are not", "can't": "cannot", "'cause": "because",
    "could've": "could have", "couldn't": "could not", "didn't": "did not",
    "doesn't": "does not", "don't": "do not", "hadn't": "had not", "hasn't": "has not",
    "haven't": "have not", "he'd": "he would", "he'll": "he will", "he's": "he is",
    "how'd": "how did", "how'd'y": "how do you", "how'll": "how will", "how's": "how is",
    "I'd": "I would", "I'd've": "I would have", "I'll": "I will", "I'll've": "I will have",
    "I'm": "I am", "I've": "I have", "i'd": "i would", "i'd've": "i would have",
    "i'll": "i will", "i'll've": "i will have", "i'm": "i am", "i've": "i have",
    "isn't": "is not", "it'd": "it would", "it'd've": "it would have", "it'll": "it will",
    "it'll've": "it will have", "it's": "it is", "let's": "let us", "ma'am": "madam",
    "mayn't": "may not", "might've": "might have", "mightn't": "might not",
    "mightn't've": "might not have", "must've": "must have", "mustn't": "must not",
    "mustn't've": "must not have", "needn't": "need not", "needn't've": "need not have",
    "o'clock": "of the clock", "oughtn't": "ought not", "oughtn't've": "ought not have",
    "shan't": "shall not", "sha'n't": "shall not", "shan't've": "shall not have",
    "she'd": "she would", "she'd've": "she would have", "she'll": "she will",
    "she'll've": "she will have", "she's": "she is", "should've": "should have",
    "shouldn't": "should not", "shouldn't've": "should not have", "so've": "so have",
    "so's": "so as", "this's": "this is", "that'd": "that would", "that'd've": "that would have",
    "that's": "that is", "there'd": "there would", "there'd've": "there would have",
    "there's": "there is", "here's": "here is", "they'd": "they would",
    "they'd've": "they would have", "they'll": "they will", "they'll've": "they will have",
    "they're": "they are", "they've": "they have", "to've": "to have", "wasn't": "was not",
    "we'd": "we would", "we'd've": "we would have", "we'll": "we will",
    "we'll've": "we will have", "we're": "we are", "we've": "we have", "weren't": "were not",
    "what'll": "what will", "what'll've": "what will have", "what're": "what are",
    "what's": "what is", "what've": "what have", "when's": "when is", "when've": "when have",
    "where'd": "where did", "where's": "where is", "where've": "where have",
    "who'll": "who will", "who'll've": "who will have", "who's": "who is", "who've": "who have",
    "why's": "why is", "why've": "why have", "will've": "will have", "won't": "will not",
    "won't've": "will not have", "would've": "would have", "wouldn't": "would not",
    "wouldn't've": "would not have", "y'all": "you all", "y'all'd": "you all would",
    "y'all'd've": "you all would have", "y'all're": "you all are", "y'all've": "you all have",
    "you'd": "you would", "you'd've": "you would have", "you'll": "you will",
    "you'll've": "you will have", "you're": "you are", "you've": "you have"
}

def clean_punctuation(text):
    """清理无意义的标点符号，保留有意义的标点"""
    if not isinstance(text, str):
        return text
    
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'[^a-zA-Z0-9\s!,.?:;\'"\-@#*/]<>', '', text)
    
    return text

def expand_contractions(text):
    """扩展文本中的缩略词并清理标点"""
    if not isinstance(text, str):
        return text
        
    pattern = re.compile(r'\b(' + '|'.join(re.escape(key) for key in CONTRACTION_MAPPING.keys()) + r')\b')
    
    def replace(match):
        return CONTRACTION_MAPPING.get(match.group(0).lower(), match.group(0))
    
    text = pattern.sub(replace, text)
    text = clean_punctuation(text)
    
    return text

def preprocess_text(text):
    """综合文本预处理函数"""
    if not isinstance(text, str):
        return ""
    
    # 扩展缩略词和清理标点
    text = expand_contractions(text)
    
    return text

def url_to_semantics(text: str) -> str:
    if not isinstance(text, str):
        return ""

    url_pattern = r'https?://[^\s/$.?#].[^\s]*'
    urls = re.findall(url_pattern, text)
    
    if not urls:
        return "" 

    all_semantics = []
    seen_semantics = set()

    for url in urls:
        url_lower = url.lower()
        
        domain_match = re.search(r"(?:https?://)?([a-z0-9\-\.]+)\.[a-z]{2,}", url_lower)
        if domain_match:
            full_domain = domain_match.group(1)
            parts = full_domain.split('.')
            for part in parts:
                if part and part not in seen_semantics and len(part) > 3: # Avoid short parts like 'www'
                    all_semantics.append(f"domain:{part}")
                    seen_semantics.add(part)

        # 2. Extract path parts
        path = re.sub(r"^(?:https?://)?[a-z0-9\.-]+\.[a-z]{2,}/?", "", url_lower)
        path_parts = [p for p in re.split(r'[/_.-]+', path) if p and p.isalnum()] # Split by common delimiters

        for part in path_parts:
            # Clean up potential file extensions or query params
            part_clean = re.sub(r"\.(html?|php|asp|jsp)$|#.*|\?.*", "", part)
            if part_clean and part_clean not in seen_semantics and len(part_clean) > 3:
                all_semantics.append(f"path:{part_clean}")
                seen_semantics.add(part_clean)

    if not all_semantics:
        return ""

    return f"\nURL Keywords: {' '.join(all_semantics)}"


def get_dataframe_to_train(data_path):
    train_dataset = pd.read_csv(f"{data_path}/train.csv") 
    test_dataset = pd.read_csv(f"{data_path}/test.csv")

    flatten = []

    flatten.append(train_dataset[["body", "rule", "subreddit","rule_violation"]].copy())

    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            col_name = f"{violation_type}_example_{i}"
            
            if col_name in train_dataset.columns:
                sub_dataset = train_dataset[[col_name, "rule", "subreddit"]].copy()
                sub_dataset = sub_dataset.rename(columns={col_name: "body"})
                sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
                
                sub_dataset.dropna(subset=['body'], inplace=True)
                sub_dataset = sub_dataset[sub_dataset['body'].str.strip().str.len() > 0]
                
                if not sub_dataset.empty:
                    flatten.append(sub_dataset)
    
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            col_name = f"{violation_type}_example_{i}"
            
            if col_name in test_dataset.columns:
                sub_dataset = test_dataset[[col_name, "rule", "subreddit"]].copy()
                sub_dataset = sub_dataset.rename(columns={col_name: "body"})
                sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
                
                sub_dataset.dropna(subset=['body'], inplace=True)
                sub_dataset = sub_dataset[sub_dataset['body'].str.strip().str.len() > 0]
                
                if not sub_dataset.empty:
                    flatten.append(sub_dataset)
    
    dataframe = pd.concat(flatten, axis=0)
    dataframe = dataframe.drop_duplicates(subset=['body', 'rule', 'subreddit'], ignore_index=True)
    dataframe.drop_duplicates(subset=['body','rule'],keep='first',inplace=True)
    
    # 在返回之前对body文本进行预处理
    dataframe['body'] = dataframe['body'].apply(preprocess_text)
    
    return dataframe.sample(frac=1, random_state=42).reset_index(drop=True)

Writing utils.py


In [2]:
%%writefile train_deberta.py
import os
import pandas as pd
import torch
import torch.nn as nn
import random
import numpy as np
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from utils import get_dataframe_to_train, url_to_semantics, preprocess_text

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class CFG:
    model_name_or_path = "/kaggle/input/huggingfacedebertav3variants/deberta-v3-base"
    data_path = "/kaggle/input/jigsaw-agile-community-rules/"
    output_dir = "./deberta_v3_small_final_model"
  
    EPOCHS = 3
    LEARNING_RATE = 2e-5  
    
    MAX_LENGTH = 512
    BATCH_SIZE = 8
    
    # 新增：损失函数权重
    ALPHA = 0.5  # 主损失权重
    BETA = 0.5   # 辅助损失权重

# 新增：多任务模型
class MultiTaskModel(nn.Module):
    def __init__(self, model_name_or_path, num_rules):
        super(MultiTaskModel, self).__init__()
        self.deberta = AutoModelForSequenceClassification.from_pretrained(
            model_name_or_path, num_labels=2
        )
        
        # 获取隐藏层维度
        hidden_size = self.deberta.config.hidden_size
        
        # 辅助分类头：每个规则的违反/不违反状态 (num_rules * 2 个类)
        self.auxiliary_classifier = nn.Linear(hidden_size, num_rules * 2)
        
    def forward(self, input_ids, attention_mask, labels=None, auxiliary_labels=None):
        # 获取主模型输出
        outputs = self.deberta.deberta(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        pooled_output = sequence_output[:, 0]  # 使用[CLS] token
        
        # 主分类任务（二分类）
        main_logits = self.deberta.classifier(self.deberta.dropout(pooled_output))
        
        # 辅助分类任务（多分类：每个规则的违反/不违反）
        auxiliary_logits = self.auxiliary_classifier(self.deberta.dropout(pooled_output))
        
        total_loss = None
        if labels is not None and auxiliary_labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            
            # 主损失：二分类
            main_loss = loss_fct(main_logits, labels)
            
            # 辅助损失：多分类（规则特定的违反/不违反）
            auxiliary_loss = loss_fct(auxiliary_logits, auxiliary_labels)
            
            # 组合损失
            total_loss = CFG.ALPHA * main_loss + CFG.BETA * auxiliary_loss
        
        return {
            'loss': total_loss,
            'logits': main_logits,
            'auxiliary_logits': auxiliary_logits
        }

class JigsawDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None, auxiliary_labels=None):
        self.encodings = encodings
        self.labels = labels
        self.auxiliary_labels = auxiliary_labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        if self.auxiliary_labels is not None:
            item['auxiliary_labels'] = torch.tensor(self.auxiliary_labels[idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

# 新增：自定义Trainer
class MultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # 只有在训练时才有标签
        labels = inputs.pop("labels", None)
        auxiliary_labels = inputs.pop("auxiliary_labels", None)
        
        outputs = model(**inputs, labels=labels, auxiliary_labels=auxiliary_labels)
        
        # 如果有损失，返回损失；否则只返回输出
        if outputs.get('loss') is not None:
            loss = outputs['loss']
            return (loss, outputs) if return_outputs else loss
        else:
            # 预测阶段，没有损失
            return outputs if return_outputs else None

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        """
        自定义预测步骤，确保正确处理预测阶段
        """
        inputs = self._prepare_inputs(inputs)
        
        with torch.no_grad():
            # 移除可能不存在的标签
            labels = inputs.pop("labels", None)
            auxiliary_labels = inputs.pop("auxiliary_labels", None)
            
            outputs = model(**inputs, labels=labels, auxiliary_labels=auxiliary_labels)
            
            # 返回主任务的logits用于预测
            logits = outputs['logits']
            
            loss = outputs.get('loss')
            
        if prediction_loss_only:
            return (loss, None, None)
        
        return (loss, logits, labels)

def create_auxiliary_labels(df):
    """创建辅助标签：每个规则分为违反和不违反两个类别"""
    # 创建规则到数字的映射
    rule_encoder = LabelEncoder()
    df['rule_encoded'] = rule_encoder.fit_transform(df['rule'])
    
    # 获取规则数量
    num_rules = df['rule_encoded'].nunique()
    
    # 创建辅助标签
    # 对于规则i：
    # - 如果违规：标签 = rule_encoded * 2 + 1 (奇数索引表示违规)
    # - 如果不违规：标签 = rule_encoded * 2 (偶数索引表示不违规)
    auxiliary_labels = []
    for _, row in df.iterrows():
        rule_id = row['rule_encoded']
        if row['rule_violation'] == 1:
            # 违规：使用奇数索引 (rule_id * 2 + 1)
            auxiliary_labels.append(rule_id * 2 + 1)
        else:
            # 不违规：使用偶数索引 (rule_id * 2)
            auxiliary_labels.append(rule_id * 2)
    
    return auxiliary_labels, rule_encoder, num_rules

def main():
    seed_everything(42)
    training_data_df = get_dataframe_to_train(CFG.data_path)
    print(f"Training dataset size: {len(training_data_df)}")
    
    # 查看数据中的规则数量
    unique_rules = training_data_df['rule'].nunique()
    print(f"Unique rules: {unique_rules}")
    print(f"Rules: {training_data_df['rule'].unique()}")

    test_df_for_prediction = pd.read_csv(f"{CFG.data_path}/test.csv")
    
    # 对训练数据进行文本预处理
    training_data_df['body_with_url'] = training_data_df['body'].apply(lambda x: preprocess_text(x) + url_to_semantics(x))
    training_data_df['input_text'] = training_data_df['rule'] + "[SEP]" + training_data_df['subreddit']+ "[SEP]" + training_data_df['body_with_url']

    # 创建辅助标签
    auxiliary_labels, rule_encoder, num_rules = create_auxiliary_labels(training_data_df)
    print(f"Auxiliary classes: {num_rules * 2} (每个规则2个状态: 违反/不违反)")
    print(f"规则编码映射: {dict(zip(rule_encoder.classes_, range(len(rule_encoder.classes_))))}")
    
    # 打印标签分布
    unique_aux_labels, counts = np.unique(auxiliary_labels, return_counts=True)
    print("辅助标签分布:")
    for label, count in zip(unique_aux_labels, counts):
        rule_id = label // 2
        is_violation = label % 2 == 1
        rule_name = rule_encoder.classes_[rule_id]
        status = "违反" if is_violation else "不违反"
        print(f"  规则 '{rule_name}' {status}: {count} 样本")

    tokenizer = AutoTokenizer.from_pretrained(CFG.model_name_or_path)
    train_encodings = tokenizer(training_data_df['input_text'].tolist(), truncation=True, padding=True, max_length=CFG.MAX_LENGTH)
    train_labels = training_data_df['rule_violation'].tolist()
    
    train_dataset = JigsawDataset(train_encodings, train_labels, auxiliary_labels)

    # 使用多任务模型
    model = MultiTaskModel(CFG.model_name_or_path, num_rules)
    
    training_args = TrainingArguments(
        output_dir=CFG.output_dir,
        num_train_epochs=CFG.EPOCHS,
        learning_rate=CFG.LEARNING_RATE,
        per_device_train_batch_size=CFG.BATCH_SIZE,
        warmup_ratio=0.1,
        weight_decay=0.01,
        report_to="none",
        save_strategy="no",
        logging_steps=1,
    )
    
    # 使用自定义Trainer
    trainer = MultiTaskTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
    )
    
    trainer.train()

    # 预测阶段 - 对测试数据也进行文本预处理
    test_df_for_prediction['body'] = test_df_for_prediction['body'].apply(preprocess_text)
    test_df_for_prediction['body_with_url'] = test_df_for_prediction['body'].apply(lambda x: x + url_to_semantics(x))
    test_df_for_prediction['input_text'] = test_df_for_prediction['rule'] + "[SEP]" + test_df_for_prediction['subreddit']+"[SEP]"+ test_df_for_prediction['body_with_url']
    
    test_encodings = tokenizer(test_df_for_prediction['input_text'].tolist(), truncation=True, padding=True, max_length=CFG.MAX_LENGTH)
    test_dataset = JigsawDataset(test_encodings)
    
    predictions = trainer.predict(test_dataset)
    # 使用主任务的预测结果
    probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=1)[:, 1].numpy()

    submission_df = pd.DataFrame({
        "row_id": test_df_for_prediction["row_id"],
        "rule_violation": probs
    })
    submission_df.to_csv("debert_submission.csv", index=False)

if __name__ == "__main__":
    main()

Writing train_deberta.py


In [3]:
!python train_deberta.py

2025-10-22 14:01:41.043941: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761141701.421645      57 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761141701.532550      57 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Training dataset size: 1875
Unique rules: 2
Rules: ['No legal advice: Do not offer or request legal advice.'
 'No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.']
Auxiliary classes: 4 (每个规则2个状态: 违反/不违反)
规则编码映射: {'No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.': 0, 'No legal advice: Do not offer or request legal advice.': 1}


In [4]:
%%writefile train_e5_bert.py
import os
import pandas as pd
import torch
import torch.nn as nn
import random
import numpy as np
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from utils import get_dataframe_to_train, url_to_semantics, preprocess_text

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class CFG:
    model_name_or_path = "/kaggle/input/e5-base-v2/transformers/01/1"
    data_path = "/kaggle/input/jigsaw-agile-community-rules/"
    output_dir = "./deberta_v3_small_final_model"
  
    EPOCHS = 3
    LEARNING_RATE = 2e-5  
    
    MAX_LENGTH = 512
    BATCH_SIZE = 8
    
    # 新增：损失函数权重
    ALPHA = 0.3  # 主损失权重
    BETA = 0.7   # 辅助损失权重

# 新增：多任务模型
class MultiTaskModel(nn.Module):
    def __init__(self, model_name_or_path, num_rules):
        super(MultiTaskModel, self).__init__()
        self.base_model = AutoModelForSequenceClassification.from_pretrained(
            model_name_or_path, num_labels=2
        )
        
        # 获取隐藏层维度
        hidden_size = self.base_model.config.hidden_size
        
        # 辅助分类头：每个规则的违反/不违反状态 (num_rules * 2 个类)
        self.auxiliary_classifier = nn.Linear(hidden_size, num_rules * 2)
        
    def forward(self, input_ids, attention_mask, labels=None, auxiliary_labels=None):
        # 获取基础模型输出 - 兼容BERT和DeBERTa
        if hasattr(self.base_model, 'bert'):
            # BERT模型 (包括e5-base-v2)
            outputs = self.base_model.bert(input_ids=input_ids, attention_mask=attention_mask)
            sequence_output = outputs.last_hidden_state
            pooled_output = sequence_output[:, 0]  # 使用[CLS] token
            
            # 主分类任务（二分类）
            main_logits = self.base_model.classifier(self.base_model.dropout(pooled_output))
            
        elif hasattr(self.base_model, 'deberta'):
            # DeBERTa模型
            outputs = self.base_model.deberta(input_ids=input_ids, attention_mask=attention_mask)
            sequence_output = outputs.last_hidden_state
            pooled_output = sequence_output[:, 0]  # 使用[CLS] token
            
            # 主分类任务（二分类）
            main_logits = self.base_model.classifier(self.base_model.dropout(pooled_output))
            
        else:
            # 其他模型类型，直接使用base_model的forward
            base_outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
            sequence_output = base_outputs.hidden_states[-1]  # 最后一层隐藏状态
            pooled_output = sequence_output[:, 0]  # 使用[CLS] token
            main_logits = base_outputs.logits
        
        # 辅助分类任务（多分类：每个规则的违反/不违反）
        if hasattr(self.base_model, 'dropout'):
            auxiliary_logits = self.auxiliary_classifier(self.base_model.dropout(pooled_output))
        else:
            # 如果没有dropout层，直接使用pooled_output
            auxiliary_logits = self.auxiliary_classifier(pooled_output)
        
        total_loss = None
        if labels is not None and auxiliary_labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            
            # 主损失：二分类
            main_loss = loss_fct(main_logits, labels)
            
            # 辅助损失：多分类（规则特定的违反/不违反）
            auxiliary_loss = loss_fct(auxiliary_logits, auxiliary_labels)
            
            # 组合损失
            total_loss = CFG.ALPHA * main_loss + CFG.BETA * auxiliary_loss
        
        return {
            'loss': total_loss,
            'logits': main_logits,
            'auxiliary_logits': auxiliary_logits
        }

class JigsawDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None, auxiliary_labels=None):
        self.encodings = encodings
        self.labels = labels
        self.auxiliary_labels = auxiliary_labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        if self.auxiliary_labels is not None:
            item['auxiliary_labels'] = torch.tensor(self.auxiliary_labels[idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

# 新增：自定义Trainer
class MultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # 只有在训练时才有标签
        labels = inputs.pop("labels", None)
        auxiliary_labels = inputs.pop("auxiliary_labels", None)
        
        outputs = model(**inputs, labels=labels, auxiliary_labels=auxiliary_labels)
        
        # 如果有损失，返回损失；否则只返回输出
        if outputs.get('loss') is not None:
            loss = outputs['loss']
            return (loss, outputs) if return_outputs else loss
        else:
            # 预测阶段，没有损失
            return outputs if return_outputs else None

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        """
        自定义预测步骤，确保正确处理预测阶段
        """
        inputs = self._prepare_inputs(inputs)
        
        with torch.no_grad():
            # 移除可能不存在的标签
            labels = inputs.pop("labels", None)
            auxiliary_labels = inputs.pop("auxiliary_labels", None)
            
            outputs = model(**inputs, labels=labels, auxiliary_labels=auxiliary_labels)
            
            # 返回主任务的logits用于预测
            logits = outputs['logits']
            
            loss = outputs.get('loss')
            
        if prediction_loss_only:
            return (loss, None, None)
        
        return (loss, logits, labels)

def create_auxiliary_labels(df):
    """创建辅助标签：每个规则分为违反和不违反两个类别"""
    # 创建规则到数字的映射
    rule_encoder = LabelEncoder()
    df['rule_encoded'] = rule_encoder.fit_transform(df['rule'])
    
    # 获取规则数量
    num_rules = df['rule_encoded'].nunique()
    
    # 创建辅助标签
    # 对于规则i：
    # - 如果违规：标签 = rule_encoded * 2 + 1 (奇数索引表示违规)
    # - 如果不违规：标签 = rule_encoded * 2 (偶数索引表示不违规)
    auxiliary_labels = []
    for _, row in df.iterrows():
        rule_id = row['rule_encoded']
        if row['rule_violation'] == 1:
            # 违规：使用奇数索引 (rule_id * 2 + 1)
            auxiliary_labels.append(rule_id * 2 + 1)
        else:
            # 不违规：使用偶数索引 (rule_id * 2)
            auxiliary_labels.append(rule_id * 2)
    
    return auxiliary_labels, rule_encoder, num_rules

def main():
    seed_everything(42)
    training_data_df = get_dataframe_to_train(CFG.data_path)
    print(f"Training dataset size: {len(training_data_df)}")
    
    # 查看数据中的规则数量
    unique_rules = training_data_df['rule'].nunique()
    print(f"Unique rules: {unique_rules}")
    print(f"Rules: {training_data_df['rule'].unique()}")

    test_df_for_prediction = pd.read_csv(f"{CFG.data_path}/test.csv")
    
    # 对训练数据进行文本预处理
    training_data_df['body_with_url'] = training_data_df['body'].apply(lambda x: preprocess_text(x) + url_to_semantics(x))
    training_data_df['input_text'] = training_data_df['rule'] + "[SEP]" + training_data_df['subreddit']+ "[SEP]" + training_data_df['body_with_url']

    # 创建辅助标签
    auxiliary_labels, rule_encoder, num_rules = create_auxiliary_labels(training_data_df)
    print(f"Auxiliary classes: {num_rules * 2} (每个规则2个状态: 违反/不违反)")
    print(f"规则编码映射: {dict(zip(rule_encoder.classes_, range(len(rule_encoder.classes_))))}")
    
    # 打印标签分布
    unique_aux_labels, counts = np.unique(auxiliary_labels, return_counts=True)
    print("辅助标签分布:")
    for label, count in zip(unique_aux_labels, counts):
        rule_id = label // 2
        is_violation = label % 2 == 1
        rule_name = rule_encoder.classes_[rule_id]
        status = "违反" if is_violation else "不违反"
        print(f"  规则 '{rule_name}' {status}: {count} 样本")

    tokenizer = AutoTokenizer.from_pretrained(CFG.model_name_or_path)
    train_encodings = tokenizer(training_data_df['input_text'].tolist(), truncation=True, padding=True, max_length=CFG.MAX_LENGTH)
    train_labels = training_data_df['rule_violation'].tolist()
    
    train_dataset = JigsawDataset(train_encodings, train_labels, auxiliary_labels)

    # 使用多任务模型
    model = MultiTaskModel(CFG.model_name_or_path, num_rules)
    
    training_args = TrainingArguments(
        output_dir=CFG.output_dir,
        num_train_epochs=CFG.EPOCHS,
        learning_rate=CFG.LEARNING_RATE,
        per_device_train_batch_size=CFG.BATCH_SIZE,
        warmup_ratio=0.1,
        weight_decay=0.01,
        report_to="none",
        save_strategy="no",
        logging_steps=1,
    )
    
    # 使用自定义Trainer
    trainer = MultiTaskTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
    )
    
    trainer.train()

    # 预测阶段 - 对测试数据也进行文本预处理
    test_df_for_prediction['body'] = test_df_for_prediction['body'].apply(preprocess_text)
    test_df_for_prediction['body_with_url'] = test_df_for_prediction['body'].apply(lambda x: x + url_to_semantics(x))
    test_df_for_prediction['input_text'] = test_df_for_prediction['rule'] + "[SEP]" + test_df_for_prediction['subreddit']+"[SEP]"+ test_df_for_prediction['body_with_url']
    
    test_encodings = tokenizer(test_df_for_prediction['input_text'].tolist(), truncation=True, padding=True, max_length=CFG.MAX_LENGTH)
    test_dataset = JigsawDataset(test_encodings)
    
    predictions = trainer.predict(test_dataset)
    # 使用主任务的预测结果
    probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=1)[:, 1].numpy()

    submission_df = pd.DataFrame({
        "row_id": test_df_for_prediction["row_id"],
        "rule_violation": probs
    })
    submission_df.to_csv("e5_bert_submission.csv", index=False)

if __name__ == "__main__":
    main()

Writing train_e5_bert.py


In [5]:
!python train_e5_bert.py

2025-10-22 14:06:53.983903: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761142014.007131     800 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761142014.013927     800 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Training dataset size: 1875
Unique rules: 2
Rules: ['No legal advice: Do not offer or request legal advice.'
 'No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.']
Auxiliary classes: 4 (每个规则2个状态: 违反/不违反)
规则编码映射: {'No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.': 0, 'No legal advice: Do not offer or request legal advice.': 1}


In [6]:
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'trl==0.21.0' 'optimum==1.27.0' 'auto-gptq==0.7.1' 'bitsandbytes==0.46.1' 'logits-processor-zoo==0.2.1' 'vllm==0.10.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'deepspeed==0.17.4' -q
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'triton==3.2.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'clean-text'
!uv pip install --system --no-index -U --no-deps --find-links='/kaggle/input/jigsaw-packages2/whls/' 'peft' 'accelerate' 'datasets'

Using Python 3.11.13 environment at: /usr
Resolved 166 packages in 311ms
Prepared 64 packages in 34.50s
Uninstalled 26 packages in 2.41s
Installed 64 packages in 4.05s
 + astor==0.8.1
 + auto-gptq==0.7.1
 + bitsandbytes==0.46.1
 + blake3==1.0.5
 + cbor2==5.7.0
 + compressed-tensors==0.10.2
 + depyf==0.19.0
 + diskcache==5.6.3
 + fastapi-cli==0.0.10
 + fastapi-cloud-cli==0.1.5
 + gekko==1.3.0
 + gguf==0.17.1
 + httptools==0.6.4
 - huggingface-hub==0.33.1
 + huggingface-hub==0.34.4
 + interegular==0.3.3
 + lark==1.2.2
 + llguidance==0.7.30
 - llvmlite==0.43.0
 + llvmlite==0.44.0
 + lm-format-enforcer==0.10.12
 + logits-processor-zoo==0.2.1
 + mistral-common==1.8.4
 + msgspec==0.19.0
 - numba==0.60.0
 + numba==0.61.2
 - nvidia-cublas-cu12==12.5.3.2
 + nvidia-cublas-cu12==12.6.4.1
 - nvidia-cuda-cupti-cu12==12.5.82
 + nvidia-cuda-cupti-cu12==12.6.80
 - nvidia-cuda-nvrtc-cu12==12.5.82
 + nvidia-cuda-nvrtc-cu12==12.6.77
 - nvidia-cuda-runtime-cu12==12.5.82
 + nvidia-cuda-runtime-cu12==12.6.7

In [7]:
%%writefile accelerate_config.yaml
compute_environment: LOCAL_MACHINE
debug: false
deepspeed_config:
  gradient_accumulation_steps: 4
  gradient_clipping: 1.0
  train_micro_batch_size_per_gpu: 4
  
  zero_stage: 2
  offload_optimizer_device: none
  offload_param_device: none
  zero3_init_flag: false
  
  stage3_gather_16bit_weights_on_model_save: false
  stage3_max_live_parameters: 1e8
  stage3_max_reuse_distance: 1e8
  stage3_prefetch_bucket_size: 5e7
  stage3_param_persistence_threshold: 1e5
  
  zero_allow_untested_optimizer: true
  zero_force_ds_cpu_optimizer: false
  
  fp16:
    enabled: true
    loss_scale: 0
    initial_scale_power: 16
    loss_scale_window: 1000
    hysteresis: 2
    min_loss_scale: 1
  
distributed_type: DEEPSPEED
downcast_bf16: 'no'
dynamo_config:
  dynamo_backend: INDUCTOR
  dynamo_use_fullgraph: false
  dynamo_use_dynamic: false
enable_cpu_affinity: false
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 2
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false

Writing accelerate_config.yaml


In [ ]:
!accelerate launch --config_file accelerate_config.yaml Model1/train.py
!python Model1/inference.py

[2025-10-22 14:12:06,083] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2025-10-22 14:12:09,774] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False
2025-10-22 14:12:10.159905: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761142330.182380    1677 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761142330.190490    1677 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W1022 14:12:14.920000 1677 torch/distributed/run.py:766] 
W1022 14:12:14.920000 1677 torch/distributed/run.py:766] *****************************************
W1022 14:12:14.920000 1677 t

In [ ]:
!accelerate launch --config_file accelerate_config.yaml Model4/train.py
!python Model4/inference.py

[2025-10-22 14:31:03,562] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2025-10-22 14:31:05,306] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False
2025-10-22 14:31:05.642306: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761143465.665423    2161 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761143465.674041    2161 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W1022 14:31:09.661000 2161 torch/distributed/run.py:766] 
W1022 14:31:09.661000 2161 torch/distributed/run.py:766] *****************************************
W1022 14:31:09.661000 2161 t

In [ ]:
!accelerate launch --config_file accelerate_config.yaml Model3/train.py
!python Model3/inference.py

[2025-10-22 14:45:30,293] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2025-10-22 14:45:33,253] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False
2025-10-22 14:45:34.124017: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761144334.145896    2540 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761144334.152714    2540 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W1022 14:45:40.879000 2540 torch/distributed/run.py:766] 
W1022 14:45:40.879000 2540 torch/distributed/run.py:766] *****************************************
W1022 14:45:40.879000 2540 t

In [ ]:
!accelerate launch --config_file accelerate_config.yaml Model2/train.py
!python Model2/inference.py

[2025-10-22 15:06:25,743] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2025-10-22 15:06:27,314] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False
2025-10-22 15:06:27.598962: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761145587.621304    2901 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761145587.628232    2901 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W1022 15:06:31.533000 2901 torch/distributed/run.py:766] 
W1022 15:06:31.533000 2901 torch/distributed/run.py:766] *****************************************
W1022 15:06:31.533000 2901 t

In [12]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizer,
    BertModel,
    get_linear_schedule_with_warmup,
    AutoModel,
    AutoConfig,
)
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
import re
import random
import os
from collections import OrderedDict

# 设置设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

COMPLETE_PHRASE = "Answer:"
BASE_PROMPT = "You are a content moderation assistant. Your task is to determine whether a given comment violates a specific rule of a subreddit."

# 定义标签文本
LABEL_TEXTS = {
    "violation": ["violation", "rule breaking", "inappropriate", "offensive", "harmful"],
    "non_violation": ["non violation", "appropriate", "acceptable", "harmless", "friendly"]
}

CONTRACTION_MAPPING = {
    "ain't": "is not", "aren't": "are not", "can't": "cannot", "'cause": "because",
    "could've": "could have", "couldn't": "could not", "didn't": "did not",
    "doesn't": "does not", "don't": "do not", "hadn't": "had not", "hasn't": "has not",
    "haven't": "have not", "he'd": "he would", "he'll": "he will", "he's": "he is",
    "how'd": "how did", "how'd'y": "how do you", "how'll": "how will", "how's": "how is",
    "I'd": "I would", "I'd've": "I would have", "I'll": "I will", "I'll've": "I will have",
    "I'm": "I am", "I've": "I have", "i'd": "i would", "i'd've": "i would have",
    "i'll": "i will", "i'll've": "i will have", "i'm": "i am", "i've": "i have",
    "isn't": "is not", "it'd": "it would", "it'd've": "it would have", "it'll": "it will",
    "it'll've": "it will have", "it's": "it is", "let's": "let us", "ma'am": "madam",
    "mayn't": "may not", "might've": "might have", "mightn't": "might not",
    "mightn't've": "might not have", "must've": "must have", "mustn't": "must not",
    "mustn't've": "must not have", "needn't": "need not", "needn't've": "need not have",
    "o'clock": "of the clock", "oughtn't": "ought not", "oughtn't've": "ought not have",
    "shan't": "shall not", "sha'n't": "shall not", "shan't've": "shall not have",
    "she'd": "she would", "she'd've": "she would have", "she'll": "she will",
    "she'll've": "she will have", "she's": "she is", "should've": "should have",
    "shouldn't": "should not", "shouldn't've": "should not have", "so've": "so have",
    "so's": "so as", "this's": "this is", "that'd": "that would", "that'd've": "that would have",
    "that's": "that is", "there'd": "there would", "there'd've": "there would have",
    "there's": "there is", "here's": "here is", "they'd": "they would",
    "they'd've": "they would have", "they'll": "they will", "they'll've": "they will have",
    "they're": "they are", "they've": "they have", "to've": "to have", "wasn't": "was not",
    "we'd": "we would", "we'd've": "we would have", "we'll": "we will",
    "we'll've": "we will have", "we're": "we are", "we've": "we have", "weren't": "were not",
    "what'll": "what will", "what'll've": "what will have", "what're": "what are",
    "what's": "what is", "what've": "what have", "when's": "when is", "when've": "when have",
    "where'd": "where did", "where's": "where is", "where've": "where have",
    "who'll": "who will", "who'll've": "who will have", "who's": "who is", "who've": "who have",
    "why's": "why is", "why've": "why have", "will've": "will have", "won't": "will not",
    "won't've": "will not have", "would've": "would have", "wouldn't": "would not",
    "wouldn't've": "would not have", "y'all": "you all", "y'all'd": "you all would",
    "y'all'd've": "you all would have", "y'all're": "you all are", "y'all've": "you all have",
    "you'd": "you would", "you'd've": "you would have", "you'll": "you will",
    "you'll've": "you will have", "you're": "you are", "you've": "you have"
}

# 公共函数定义
def clean_punctuation(text):
    """清理无意义的标点符号，保留有意义的标点"""
    if not isinstance(text, str):
        return text
    
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'[^a-zA-Z0-9\s!,.?:;\'"\-@#*/]<>', '', text)
    
    return text

def expand_contractions(text):
    """扩展文本中的缩略词并清理标点"""
    if not isinstance(text, str):
        return text
        
    pattern = re.compile(r'\b(' + '|'.join(re.escape(key) for key in CONTRACTION_MAPPING.keys()) + r')\b')
    
    def replace(match):
        return CONTRACTION_MAPPING.get(match.group(0).lower(), match.group(0))
    
    text = pattern.sub(replace, text)
    text = clean_punctuation(text)
    
    return text

def get_dataframe_to_train(data_path, use_external_data=False):
    """数据加载函数，支持外部数据集"""
    train_dataset = pd.read_csv(f"{data_path}/train.csv")
    test_dataset = pd.read_csv(f"{data_path}/test.csv")
    
    if use_external_data:
        try:
            extr_dataset = pd.read_csv("/kaggle/input/myopenaidata/myopenaidata.csv")
            extr_dataset['body'] = extr_dataset['body'].apply(expand_contractions)
            wanted_rules = [
                'No promotion of illegal activity: Do not encourage or promote illegal activities, such as drug-related activity, violence, exploitation, theft, or other criminal behavior.',
                'No medical advice: Do not offer or request specific medical advice, diagnoses, or treatment recommendations.',
                'No financial advice: We do not permit comments that make personal recommendations for investments, taxes, or careers.'
            ]
            extr_dataset = extr_dataset[extr_dataset["rule"].isin(wanted_rules)]

            print(set(extr_dataset['rule']))
        except FileNotFoundError:
            print("External dataset not found, proceeding without it...")
            extr_dataset = None
    else:
        extr_dataset = None
    
    train_dataset["body"] = train_dataset["body"].apply(expand_contractions)
    for i in range(1, 3):
        test_dataset[f"positive_example_{i}"] = test_dataset[f"positive_example_{i}"].apply(expand_contractions)
        test_dataset[f"negative_example_{i}"] = test_dataset[f"negative_example_{i}"].apply(expand_contractions)

    flatten = []
    flatten.append(train_dataset[["body", "rule", "subreddit", "rule_violation"]])
    
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            sub_dataset = test_dataset[[f"{violation_type}_example_{i}", "rule", "subreddit"]].copy()
            sub_dataset = sub_dataset.rename(columns={f"{violation_type}_example_{i}": "body"})
            sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
            flatten.append(sub_dataset)

    dataframe = pd.concat(flatten, axis=0)
    dataframe = dataframe.drop_duplicates(ignore_index=True)
    
    if extr_dataset is not None:
        dataframe = pd.concat([dataframe, extr_dataset], axis=0)
    
    return dataframe

def get_tfidf_keywords(texts, n_keywords=5):
    """从文本中提取TF-IDF最高的关键词"""
    vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(texts)
    feature_names = vectorizer.get_feature_names_out()
    
    avg_tfidf = tfidf_matrix.mean(axis=0).A1
    top_indices = avg_tfidf.argsort()[-n_keywords:][::-1]
    keywords = [feature_names[i] for i in top_indices]
    
    return keywords

def enhance_label_texts(dataframe, n_keywords=5):
    """为每个类别添加TF-IDF关键词"""
    enhanced_labels = LABEL_TEXTS.copy()
    
    violation_texts = dataframe[dataframe["rule_violation"] == 1]["body"].tolist()
    non_violation_texts = dataframe[dataframe["rule_violation"] == 0]["body"].tolist()
    
    if violation_texts:
        violation_keywords = get_tfidf_keywords(violation_texts, n_keywords)
        enhanced_labels["violation"].extend(violation_keywords)
    
    if non_violation_texts:
        non_violation_keywords = get_tfidf_keywords(non_violation_texts, n_keywords)
        enhanced_labels["non_violation"].extend(non_violation_keywords)
    
    return enhanced_labels

def build_prompt(row, enhanced_labels=None, use_sep=True):
    """构建包含标签嵌入的提示文本"""
    if enhanced_labels is None:
        enhanced_labels = LABEL_TEXTS
    
    violation_labels = " ".join(enhanced_labels["violation"])
    non_violation_labels = " ".join(enhanced_labels["non_violation"])
    all_labels = f"Violation indicators: {violation_labels}. Non-violation indicators: {non_violation_labels}"
    
    if use_sep:
        return f"""
{BASE_PROMPT}

r/subreddit: {row["subreddit"]} rule: {row["rule"]}

Labels: {all_labels} [SEP] Comment: {row["body"]}
---
{COMPLETE_PHRASE}"""
    else:
        return f"""
{BASE_PROMPT}

r/subreddit: {row["subreddit"]} rule: {row["rule"]}

Labels: {all_labels} Comment: {row["body"]}
---
{COMPLETE_PHRASE}"""

class RedditCommentDataset(Dataset):
    """自定义数据集类"""
    def __init__(self, dataframe, tokenizer, rule_mapping, max_length=512, use_sep=True, enhanced_labels=None):
        self.dataframe = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.use_sep = use_sep
        self.enhanced_labels = enhanced_labels
        self.rule_mapping = rule_mapping
        
        self.texts = [build_prompt(row, enhanced_labels, use_sep) for _, row in dataframe.iterrows()]
        
        if "rule_violation" in dataframe:
            self.labels = self.create_multi_labels(dataframe)
        else:
            self.labels = None
        
        if "rule" in dataframe:
            self.rule_indices = [self.rule_mapping[rule] for rule in dataframe["rule"]]
        else:
            self.rule_indices = None
        
    def create_multi_labels(self, dataframe):
        """创建多标签矩阵"""
        labels = np.zeros((len(dataframe), len(self.rule_mapping)), dtype=np.float32)
        for i, (_, row) in enumerate(dataframe.iterrows()):
            rule_idx = self.rule_mapping[row["rule"]]
            labels[i, rule_idx] = row["rule_violation"]
        return labels
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        item = {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "token_type_ids": encoding["token_type_ids"].flatten() if "token_type_ids" in encoding else torch.zeros_like(encoding["input_ids"].flatten())
        }
        
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        
        if self.rule_indices is not None:
            item["rule_index"] = torch.tensor(self.rule_indices[idx], dtype=torch.long)
            
        if "row_id" in self.dataframe.columns:
            item["row_id"] = torch.tensor(self.dataframe.iloc[idx]["row_id"], dtype=torch.long)
            
        return item

class FGM:
    """Fast Gradient Method (FGM)对抗训练"""
    def __init__(self, model, epsilon=0.5):
        self.model = model
        self.epsilon = epsilon
        self.backup = {}

    def attack(self, embedding_name='bert.embeddings.word_embeddings'):
        """在embedding上添加对抗扰动"""
        for name, param in self.model.named_parameters():
            if param.requires_grad and embedding_name in name:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    r_at = self.epsilon * param.grad / norm
                    param.data.add_(r_at)

    def restore(self, embedding_name='bert.embeddings.word_embeddings'):
        """恢复原始的embedding参数"""
        for name, param in self.model.named_parameters():
            if param.requires_grad and embedding_name in name:
                if name in self.backup:
                    param.data = self.backup[name]
        self.backup = {}

def set_seed(seed=42):
    """设置随机种子确保实验可复现"""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def seed_worker(worker_id):
    """DataLoader worker种子设置"""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# ==================== 模型定义 ====================

class TextCNN(nn.Module):
    """TextCNN模块"""
    def __init__(self, hidden_size=768, encode_layer=12, filter_sizes=[2, 3, 4], num_filters=128, n_class=2):
        super(TextCNN, self).__init__()
        self.num_filter_total = num_filters * len(filter_sizes)
        self.Weight = nn.Linear(self.num_filter_total, n_class, bias=False)
        self.bias = nn.Parameter(torch.ones([n_class]))
        self.filter_list = nn.ModuleList([
            nn.Conv2d(1, num_filters, kernel_size=(size, hidden_size)) for size in filter_sizes
        ])
        self.encode_layer = encode_layer
        self.filter_sizes = filter_sizes

    def forward(self, x):
        x = x.unsqueeze(1)

        pooled_outputs = []
        for i, conv in enumerate(self.filter_list):
            conv_out_height = self.encode_layer - self.filter_sizes[i] + 1
            
            h = F.relu(conv(x))
            mp = nn.MaxPool2d(kernel_size=(conv_out_height, 1))
            pooled = mp(h).permute(0, 3, 2, 1)
            pooled_outputs.append(pooled)

        h_pool = torch.cat(pooled_outputs, dim=3)
        h_pool_flat = torch.reshape(h_pool, [-1, self.num_filter_total])

        output = self.Weight(h_pool_flat) + self.bias
        return output

class BertBlendCNN(nn.Module):
    """模型3: BERT+TextCNN混合模型"""
    def __init__(self, model_name, n_rules):
        super(BertBlendCNN, self).__init__()
        config = AutoConfig.from_pretrained(model_name)
        self.hidden_size = config.hidden_size
        self.num_hidden_layers = config.num_hidden_layers
        self.n_rules = n_rules
        
        self.bert = AutoModel.from_pretrained(model_name, output_hidden_states=True, return_dict=True)
        self.textcnn = TextCNN(
            hidden_size=self.hidden_size,
            encode_layer=self.num_hidden_layers,
            n_class=n_rules
        )

    def forward(self, input_ids, attention_mask, token_type_ids):
        outputs = self.bert(
            input_ids=input_ids, 
            attention_mask=attention_mask, 
            token_type_ids=token_type_ids
        )
        
        hidden_states = outputs.hidden_states
        
        cls_embeddings = hidden_states[1][:, 0, :].unsqueeze(1)
        
        for i in range(2, self.num_hidden_layers + 1):
            cls_embeddings = torch.cat(
                (cls_embeddings, hidden_states[i][:, 0, :].unsqueeze(1)), dim=1
            )
        
        logits = self.textcnn(cls_embeddings)
        return logits

class CombinedModel(nn.Module):
    """模型2: 组合模型（BERT+FastText+TextCNN）（增加辅助分类器）"""
    def __init__(self, model_name, n_rules, use_fasttext=True, use_textcnn=True, 
                 d_fasttext=300, n_heads=8, dropout=0.4):
        super(CombinedModel, self).__init__()
        config = AutoConfig.from_pretrained(model_name)
        self.hidden_size = config.hidden_size
        self.num_hidden_layers = config.num_hidden_layers
        self.n_rules = n_rules
        self.use_fasttext = use_fasttext
        self.use_textcnn = use_textcnn
        
        self.bert = AutoModel.from_pretrained(model_name, output_hidden_states=True, return_dict=True)
        
        if self.use_fasttext:
            self.fasttext_embedding = nn.Embedding(config.vocab_size, d_fasttext)
            self.linear_fasttext = nn.Linear(d_fasttext, self.hidden_size)
            
            self.feature_filter = nn.MultiheadAttention(
                embed_dim=self.hidden_size, 
                num_heads=n_heads, 
                dropout=dropout, 
                batch_first=True
            )
            
            self.feature_fusion = nn.MultiheadAttention(
                embed_dim=self.hidden_size, 
                num_heads=n_heads, 
                dropout=dropout, 
                batch_first=True
            )
            
            self.linear_combine = nn.Linear(self.hidden_size * 2, self.hidden_size)
        
        if self.use_textcnn:
            self.textcnn = TextCNN(
                hidden_size=self.hidden_size,
                encode_layer=self.num_hidden_layers,
                n_class=self.hidden_size
            )
        
        self.dropout = nn.Dropout(dropout)
        
        classifier_input_size = self.hidden_size
        if self.use_fasttext:
            classifier_input_size += self.hidden_size
        if self.use_textcnn:
            classifier_input_size += self.hidden_size
        
        # 主分类器：每个规则的违规/不违规    
        self.classifier = nn.Linear(classifier_input_size, n_rules)
        # 辅助分类器：粗多标签分类（n_rules个违规类别 + 1个不违规类别）
        self.auxiliary_classifier = nn.Linear(classifier_input_size, n_rules + 1)

    def forward(self, input_ids, attention_mask, token_type_ids):
        outputs = self.bert(
            input_ids=input_ids, 
            attention_mask=attention_mask, 
            token_type_ids=token_type_ids
        )
        
        hidden_states = outputs.hidden_states
        pooled_output = outputs.pooler_output
        last_hidden_state = outputs.last_hidden_state
        
        features = [pooled_output]
        
        if self.use_fasttext:
            batch_size, seq_len = input_ids.shape
            
            fasttext_embeds = self.fasttext_embedding(input_ids)
            d_fasttext = fasttext_embeds.shape[-1]
            padded = torch.cat([torch.zeros(batch_size, 1, d_fasttext, device=input_ids.device), 
                               fasttext_embeds, 
                               torch.zeros(batch_size, 1, d_fasttext, device=input_ids.device)], dim=1)
            fasttext_vectors = (padded[:, :-2] + padded[:, 2:]) / 2
            
            fasttext_transformed = self.linear_fasttext(fasttext_vectors)
            
            filtered_features, _ = self.feature_filter(
                query=fasttext_transformed,
                key=last_hidden_state,
                value=last_hidden_state,
                key_padding_mask=~attention_mask.bool()
            )
            
            pooled_expanded = pooled_output.unsqueeze(1).expand(-1, seq_len, -1)
            combined_features = torch.cat([filtered_features, last_hidden_state], dim=-1)
            combined_features = self.linear_combine(combined_features)
            
            fused_features, _ = self.feature_fusion(
                query=pooled_expanded,
                key=combined_features,
                value=combined_features,
                key_padding_mask=~attention_mask.bool()
            )
            
            fasttext_features = fused_features.mean(dim=1)
            features.append(fasttext_features)
        
        if self.use_textcnn:
            cls_embeddings = hidden_states[1][:, 0, :].unsqueeze(1)
            
            for i in range(2, self.num_hidden_layers + 1):
                cls_embeddings = torch.cat(
                    (cls_embeddings, hidden_states[i][:, 0, :].unsqueeze(1)), dim=1
                )
            
            textcnn_features = self.textcnn(cls_embeddings)
            features.append(textcnn_features)
        
        final_features = torch.cat(features, dim=-1)
        
        # 主分类器输出
        main_logits = self.classifier(self.dropout(final_features))
        # 辅助分类器输出
        aux_logits = self.auxiliary_classifier(self.dropout(final_features))
        
        return main_logits, aux_logits

def create_auxiliary_labels(labels, rule_indices, n_rules):
    """
    创建粗多标签分类的标签
    如果有违规，则对应规则位置为1，其余为0
    如果没有违规，则最后一位（不违规类别）为1，其余为0
    """
    batch_size = labels.shape[0]
    aux_labels = torch.zeros(batch_size, n_rules + 1, device=labels.device)
    
    for i in range(batch_size):
        rule_idx = rule_indices[i].item()
        if labels[i, rule_idx].item() > 0.5:  # 违规
            aux_labels[i, rule_idx] = 1.0
        else:  # 不违规
            aux_labels[i, n_rules] = 1.0  # 最后一位是不违规类别
    
    return aux_labels

# ==================== 训练和预测函数 ====================

def train_combined_model(model, train_loader, epochs=3, learning_rate=2e-5,
                        multi_label_weight=0.4, binary_weight=0.4, aux_weight=0.2, 
                        use_adversarial=True):
    """训练组合模型（使用全部数据）"""
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=learning_rate)
    
    multi_label_loss_fn = nn.BCEWithLogitsLoss()
    binary_loss_fn = nn.BCELoss()
    aux_loss_fn = nn.CrossEntropyLoss()  # 辅助分类器使用交叉熵损失
    
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps
    )
    
    if use_adversarial:
        fgm = FGM(model, epsilon=0.5)
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_main_loss = 0
        total_binary_loss = 0
        total_aux_loss = 0
        
        progress_bar = tqdm(train_loader, desc=f"CombinedModel Epoch {epoch+1}/{epochs}")
        
        for batch in progress_bar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)
            labels = batch["labels"].to(device).float()
            rule_indices = batch["rule_index"].to(device)
            
            optimizer.zero_grad()
            
            main_logits, aux_logits = model(input_ids, attention_mask, token_type_ids)
            
            # 主要损失：多标签分类
            multi_label_loss = multi_label_loss_fn(main_logits, labels)
            
            # 二分类损失：特定规则的违规/不违规
            rule_logits = main_logits.gather(1, rule_indices.view(-1, 1)).squeeze(1)
            rule_labels = labels.gather(1, rule_indices.view(-1, 1)).squeeze(1)
            rule_probs = torch.sigmoid(rule_logits)
            binary_loss = binary_loss_fn(rule_probs, rule_labels)
            
            # 辅助损失：粗多标签分类
            aux_labels = create_auxiliary_labels(labels, rule_indices, model.n_rules)
            aux_target = torch.argmax(aux_labels, dim=1)  # 转换为类别索引
            aux_loss = aux_loss_fn(aux_logits, aux_target)
            
            # 总损失
            loss = (multi_label_weight * multi_label_loss + 
                   binary_weight * binary_loss + 
                   aux_weight * aux_loss)
            
            loss.backward()
            
            # 对抗训练
            if use_adversarial:
                fgm.attack()
                main_logits_adv, aux_logits_adv = model(input_ids, attention_mask, token_type_ids)
                multi_label_loss_adv = multi_label_loss_fn(main_logits_adv, labels)
                rule_logits_adv = main_logits_adv.gather(1, rule_indices.view(-1, 1)).squeeze(1)
                rule_probs_adv = torch.sigmoid(rule_logits_adv)
                binary_loss_adv = binary_loss_fn(rule_probs_adv, rule_labels)
                aux_loss_adv = aux_loss_fn(aux_logits_adv, aux_target)
                
                loss_adv = (multi_label_weight * multi_label_loss_adv + 
                           binary_weight * binary_loss_adv + 
                           aux_weight * aux_loss_adv)
                loss_adv.backward()
                fgm.restore()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()
            total_main_loss += multi_label_loss.item()
            total_binary_loss += binary_loss.item()
            total_aux_loss += aux_loss.item()
            
            progress_bar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "main": f"{multi_label_loss.item():.4f}",
                "binary": f"{binary_loss.item():.4f}",
                "aux": f"{aux_loss.item():.4f}"
            })
        
        avg_loss = total_loss / len(train_loader)
        avg_main_loss = total_main_loss / len(train_loader)
        avg_binary_loss = total_binary_loss / len(train_loader)
        avg_aux_loss = total_aux_loss / len(train_loader)
        
        print(f"CombinedModel Epoch {epoch+1} - Total: {avg_loss:.4f}, "
              f"Main: {avg_main_loss:.4f}, Binary: {avg_binary_loss:.4f}, "
              f"Aux: {avg_aux_loss:.4f}")
    
    return model

def predict(model, test_loader, rule_mapping):
    """预测函数（更新以处理双输出）"""
    model.eval()
    predictions = []
    row_ids = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Predicting"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)
            rule_indices = batch["rule_index"].to(device)
            
            main_logits, aux_logits = model(input_ids, attention_mask, token_type_ids)
            main_probs = torch.sigmoid(main_logits)
            
            batch_predictions = main_probs.gather(1, rule_indices.view(-1, 1)).squeeze(1)
            predictions.extend(batch_predictions.cpu().numpy())
            
            if "row_id" in batch:
                row_ids.extend(batch["row_id"].cpu().numpy())
    
    return predictions, row_ids

def ensemble_predictions(predictions_list, weights=[0.2, 0.2, 0.3,0.3]):
    """集成多个模型的预测结果"""
    ensemble_pred = np.zeros_like(predictions_list[0])
    for i, pred in enumerate(predictions_list):
        ensemble_pred += np.array(pred) * weights[i]
    
    return ensemble_pred

set_seed(42)

# 设置路径和参数
data_path = "/kaggle/input/jigsaw-agile-community-rules"
model_name = "/kaggle/input/baai/transformers/bge-small-en-v1.5/1"

# 加载数据
train_df = get_dataframe_to_train(data_path, use_external_data=True)

# 创建规则映射
unique_rules = train_df["rule"].unique()
rule_mapping = {rule: idx for idx, rule in enumerate(unique_rules)}
n_rules = len(unique_rules)

print(f"Number of rules: {n_rules}")
print(f"Training samples: {len(train_df)}")

# 增强标签文本
enhanced_labels = enhance_label_texts(train_df, n_keywords=5)
print("Enhanced labels:", enhanced_labels)

# 使用全部训练数据，不分割验证集
train_data = train_df  # 直接使用全部数据

# 初始化tokenizer
tokenizer = BertTokenizer.from_pretrained(model_name)

# 创建数据集（使用全部训练数据）
train_dataset = RedditCommentDataset(
    train_data, tokenizer, rule_mapping, max_length=256, use_sep=False, enhanced_labels=enhanced_labels
)

# 创建数据加载器
g = torch.Generator()
g.manual_seed(42)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, worker_init_fn=seed_worker, generator=g)
# ==================== 训练模型（重新组织顺序） ====================

models = {}
model_predictions = {}

# 首先训练 CombinedModel
print("\n" + "="*50)
print("Training Combined Model (Full Data)")
print("="*50)
combined_model = CombinedModel(
    model_name=model_name,
    n_rules=n_rules,
    use_fasttext=True,
    use_textcnn=True,
    d_fasttext=300,
    n_heads=8,
    dropout=0.4
)

trained_combined = train_combined_model(
    combined_model, train_loader, epochs=3, learning_rate=3e-5,
    multi_label_weight=0.4, binary_weight=0.4, aux_weight=0.2, use_adversarial=True
)

print("\n" + "="*50)
print("Making Predictions on Test Set")
print("="*50)

test_df = pd.read_csv(f"{data_path}/test.csv")
test_dataset = RedditCommentDataset(
    test_df, tokenizer, rule_mapping, max_length=256, use_sep=False, enhanced_labels=enhanced_labels
)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, worker_init_fn=seed_worker, generator=g)
predictions, row_ids = predict(combined_model, test_loader, rule_mapping)
# 保存单个模型的预测结果
single_submission = pd.DataFrame({
    "row_id": row_ids,
    "rule_violation": predictions
})
single_submission.to_csv("submission_combine_bert.csv", index=False)

2025-10-22 15:25:50.138304: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761146750.162042      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761146750.169021      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


{'No medical advice: Do not offer or request specific medical advice, diagnoses, or treatment recommendations.', 'No financial advice: We do not permit comments that make personal recommendations for investments, taxes, or careers.', 'No promotion of illegal activity: Do not encourage or promote illegal activities, such as drug-related activity, violence, exploitation, theft, or other criminal behavior.'}
Number of rules: 5
Training samples: 5562
Enhanced labels: {'violation': ['violation', 'rule breaking', 'inappropriate', 'offensive', 'harmful', 'need', 'com', 'http', 'person', 'just'], 'non_violation': ['non violation', 'appropriate', 'acceptable', 'harmless', 'friendly', 'person', 'com', 'organization', 'http', 'like']}

Training Combined Model (Full Data)


CombinedModel Epoch 1/3: 100%|██████████| 174/174 [03:04<00:00,  1.06s/it, loss=0.3137, main=0.1524, binary=0.3673, aux=0.5289]


CombinedModel Epoch 1 - Total: 0.5723, Main: 0.3429, Binary: 0.5726, Aux: 1.0304


CombinedModel Epoch 2/3: 100%|██████████| 174/174 [03:03<00:00,  1.05s/it, loss=0.2339, main=0.1102, binary=0.3079, aux=0.3332]


CombinedModel Epoch 2 - Total: 0.2432, Main: 0.1208, Binary: 0.3061, Aux: 0.3623


CombinedModel Epoch 3/3: 100%|██████████| 174/174 [03:02<00:00,  1.05s/it, loss=0.2594, main=0.0991, binary=0.3670, aux=0.3650]


CombinedModel Epoch 3 - Total: 0.1806, Main: 0.0815, Binary: 0.2397, Aux: 0.2603

Making Predictions on Test Set


Predicting: 100%|██████████| 1/1 [00:00<00:00, 13.66it/s]


In [13]:
import pandas as pd
import numpy as np # Ensure numpy is imported for the calculations

def ensemble_predictions(predictions_list, weights=[0.1, 0.1, 0.2, 0.2, 0.4]):
    """
    Performs a weighted average ensemble of predictions.

    Args:
        predictions_list (list of lists or arrays): A list where each element
            is the list of predictions from one model.
        weights (list): A list of weights, corresponding to the order of 
            models in predictions_list.

    Returns:
        np.array: The final ensemble prediction (weighted average).
    """
    predictions_array = np.array(predictions_list)
    
    weights_array = np.array(weights)
    
    ensemble_pred = (predictions_array.T * weights_array).sum(axis=1)
    
    return ensemble_pred

debert_res = pd.read_csv("/kaggle/working/debert_submission.csv")
debert_res_pre = list(debert_res['rule_violation'])

e5_bert_res = pd.read_csv("/kaggle/working/e5_bert_submission.csv")
e5_bert_res_pre = list(e5_bert_res['rule_violation'])

combine_bert_res = pd.read_csv("/kaggle/working/submission_combine_bert.csv")
combine_bert_res_pre = list(combine_bert_res['rule_violation'])

llama_res = pd.read_csv("/kaggle/working/llama_submission.csv")
llama_res_pre = list(llama_res['rule_violation'])

Qwen3_res = pd.read_csv("/kaggle/working/Qwen3_submission.csv")
Qwen3_res_pre = list(Qwen3_res['rule_violation'])

Qwen3_4B_7_res = pd.read_csv("/kaggle/working/Qwen3_4B_7_submission.csv")
Qwen3_4B_7_res_pre = list(Qwen3_4B_7_res['rule_violation'])

Qwen25_xx_res = pd.read_csv("/kaggle/working/Qwen2_5_submission.csv")
Qwen25_xx_res_pre = list(Qwen25_xx_res['rule_violation'])

all_predictions = []

all_predictions.append(debert_res_pre)    # 0.912
all_predictions.append(e5_bert_res_pre)   # 0.911
all_predictions.append(combine_bert_res_pre) # 0.912
all_predictions.append(Qwen25_xx_res_pre)  # 0.913
all_predictions.append(llama_res_pre)   # 0.920
all_predictions.append(Qwen3_4B_7_res_pre) # 0.923
all_predictions.append(Qwen3_res_pre) # 0.920


row_ids = debert_res['row_id'] # Use the row_id from one of the submission files
print("\n" + "="*50)
print("Ensembling Predictions")
print("="*50)

# weights = [0.1422, 0.1421, 0.1422, 0.1424, 0.1435, 0.1440, 0.1435]
# weights = [0.0643, 0.0551, 0.0771, 0.0964, 0.1929, 0.3857, 0.1285]
# weights = [0.1416, 0.1413, 0.1416, 0.1420, 0.1442, 0.1451, 0.1442]
weights = [0.125, 0.124, 0.125, 0.127, 0.134, 0.133, 0.132]
# 加权平均融合
ensemble_pred = ensemble_predictions(all_predictions, weights=weights)

# 保存融合后的结果
ensemble_submission = pd.DataFrame({
    "row_id": row_ids,
    "rule_violation": ensemble_pred
})
ensemble_submission.to_csv("submission.csv", index=False)

print("Ensemble submission saved to submission.csv")
print(f"Number of predictions: {len(ensemble_pred)}")
print(f"Prediction range: [{ensemble_pred.min():.4f}, {ensemble_pred.max():.4f}]")


Ensembling Predictions
Ensemble submission saved to submission.csv
Number of predictions: 10
Prediction range: [0.0216, 0.8270]
